# LeetCode #1444: Count Subarrays With Fixed Bounds

https://leetcode.com/problems/count-subarrays-with-fixed-bounds/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(n^3)$ | $O(1)$ |
| **Optimal: Sliding Window with Position Tracking ★** | $O(n)$ | $O(1)$ |

---

## Understanding the Methods

### Brute Force
Enumerate all $O(n^2)$ subarrays and verify min/max for each in $O(n)$. $O(n^3)$ total — unacceptable for $n = 10^5$.

### Optimal: Sliding Window with Position Tracking ★
One left-to-right scan maintains three indices: `badIdx` (last position of an out-of-range element), `minIdx` (last position of `minK`), `maxIdx` (last position of `maxK`). For each right endpoint `i`, valid subarrays ending at `i` have their left endpoint in the range `(badIdx, min(minIdx, maxIdx)]`. The count contribution is `max(0, min(minIdx, maxIdx) - badIdx)`. Single pass, $O(1)$ space.

**Constraints:**
* $2 \leq \text{nums.length} \leq 10^5$
* $1 \leq \text{minK} \leq \text{maxK} \leq 10^6$
* $1 \leq \text{nums}[i] \leq 10^6$


## Solutions

### C#

In [ ]:
public class Solution {
    public long CountSubarrays(int[] nums, int minK, int maxK) {
        long count = 0;
        // badIdx: last index where an out-of-range element broke the window
        int badIdx = -1, minIdx = -1, maxIdx = -1;
        for (int i = 0; i < nums.Length; i++) {
            // An out-of-range element invalidates all subarrays that include it
            if (nums[i] < minK || nums[i] > maxK) badIdx = i;
            if (nums[i] == minK) minIdx = i;
            if (nums[i] == maxK) maxIdx = i;
            // Left endpoints in (badIdx, min(minIdx,maxIdx)] yield valid fixed-bound subarrays
            count += Math.Max(0, Math.Min(minIdx, maxIdx) - badIdx);
        }
        return count;
    }
}

### Python

In [ ]:
class Solution:
    def count_subarrays(self, nums: list[int], min_k: int, max_k: int) -> int:
        count = 0
        # bad_idx: last index where an out-of-range element broke the window
        bad_idx = min_idx = max_idx = -1
        for i, v in enumerate(nums):
            # An out-of-range element invalidates all subarrays that include it
            if v < min_k or v > max_k: bad_idx = i
            if v == min_k: min_idx = i
            if v == max_k: max_idx = i
            # Left endpoints in (bad_idx, min(min_idx,max_idx)] yield valid fixed-bound subarrays
            count += max(0, min(min_idx, max_idx) - bad_idx)
        return count

### Go

In [ ]:
func countSubarrays(nums []int, minK, maxK int) int64 {
    var count int64
    // badIdx: last index where an out-of-range element broke the window
    badIdx, minIdx, maxIdx := -1, -1, -1
    for i, v := range nums {
        // An out-of-range element invalidates all subarrays that include it
        if v < minK || v > maxK { badIdx = i }
        if v == minK { minIdx = i }
        if v == maxK { maxIdx = i }
        // Left endpoints in (badIdx, min(minIdx,maxIdx)] yield valid fixed-bound subarrays
        if lo := min(minIdx, maxIdx); lo > badIdx {
            count += int64(lo - badIdx)
        }
    }
    return count
}

func min(a, b int) int {
    if a < b { return a }
    return b
}

### Rust

In [ ]:
impl Solution {
    pub fn count_subarrays(nums: Vec<i32>, min_k: i32, max_k: i32) -> i64 {
        let (mut count, mut bad_idx, mut min_idx, mut max_idx) = (0i64, -1i64, -1i64, -1i64);
        for (i, &v) in nums.iter().enumerate() {
            let i = i as i64;
            // An out-of-range element invalidates all subarrays that include it
            if v < min_k || v > max_k { bad_idx = i; }
            if v == min_k { min_idx = i; }
            if v == max_k { max_idx = i; }
            // Left endpoints in (bad_idx, min(min_idx,max_idx)] yield valid fixed-bound subarrays
            let lo = min_idx.min(max_idx);
            if lo > bad_idx { count += lo - bad_idx; }
        }
        count
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums = [1,3,3,2,2,3,1]`, `minK = 1`, `maxK = 3`
No out-of-range elements (`badIdx` stays $-1$). `minIdx` and `maxIdx` advance as 1's and 3's appear. At `i=6`: `min(6,5)=5`, contribution $= 5 - (-1) = 6$. Total: **9**.

### 2. Slightly Complex
**Input:** `nums = [1,1,1,1]`, `minK = 1`, `maxK = 1`
`minK == maxK`. Both `minIdx` and `maxIdx` update to the same index. Each position contributes `i - (-1) = i+1`. Total: $1+2+3+4 = 10$.

### 3. Edge Case: Time Factor
**Input:** $n = 10^5$, alternating `minK` and `maxK` with no out-of-range values
`badIdx` never advances; `minIdx`/`maxIdx` swap each step. Contribution is computed for all $n$ positions — single pass covers every element exactly once.

### 4. Edge Case: Space Factor
**Input:** Any `nums` of length $10^5$
Only four integer scalars (`count`, `badIdx`, `minIdx`, `maxIdx`) plus the loop index — $O(1)$ space regardless of array size or answer magnitude (which can reach $\approx n^2 / 2 \approx 2.5 \times 10^9$, requiring a 64-bit counter).

### 5. Almost-Impossible but Plausible
**Input:** `nums = [1,2,1,2,5]`, `minK = 1`, `maxK = 2`
At `i=4`: `nums[4]=5 > maxK=2`, so `badIdx=4`. Both `minIdx=3` and `maxIdx=3` are before `badIdx`. `min(3,3)-4 = -1 < 0` — contribution 0. The out-of-range element at the tail kills all subarrays reaching it, precisely captured by `max(0, ...)` guard.
